### LLM Chain 만들기


## 1. 환경 구성

### 1) 라이브러리 설치

In [ ]:
# uv add python-dotenv langchain langchain-openai

### 2) OpenAI 인증키 설정
https://openai.com/

In [1]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print(GROQ_API_KEY[:3])

gsk


In [2]:
import langchain
print(langchain.__version__)

1.2.15


In [3]:
from langchain_openai import ChatOpenAI

# Groq(OpenAI 호환) 공통 설정 — 모델/온도 변경은 이곳만 수정
GROQ_BASE_URL = "https://api.groq.com/openai/v1"  # Groq API 엔드포인트
GROQ_MODEL = "openai/gpt-oss-120b"  # 대안: "qwen/qwen3.8-27b"
MOVIE_MODEL = "qwen/qwen3.8-27b"  # 영화 추천 예제용 
TEMPERATURE = 0.7


def get_llm(model: str = GROQ_MODEL, temperature: float = TEMPERATURE) -> ChatOpenAI:
    """Groq 엔드포인트를 사용하는 ChatOpenAI 인스턴스를 생성합니다."""
    return ChatOpenAI(
        api_key=GROQ_API_KEY,
        base_url=GROQ_BASE_URL,
        model=model,
        temperature=temperature,
    )


# 기본 llm (이후 셀에서 재사용)
llm = get_llm()

## 2. LLM Chain

### 1) Prompt + LLM


In [5]:
# model: 공통 설정 셀의 llm 사용

# chain 실행
result = llm.invoke("인공지능 모델의 학습 원리에 대하여 쉽게 설명해 주세요.")
print(type(result))
print(result)

<class 'langchain_core.messages.ai.AIMessage'>
content="## 인공지능 모델(특히 딥러닝) 학습이 어떻게 이루어지는지 쉽게 풀어볼게요  \n\n### 1. 목표는 “무엇을 알게 만들고 싶은가?”\n예를 들어 **고양이 사진을 보면 “고양이”라고 말할 수 있게** 하고 싶다고 해봅시다.  \n이때 모델에게 **‘입력(사진) → 출력(고양이/고양이 아님)’**이라는 관계를 배우게 해야 합니다.\n\n---\n\n## 2. 학습에 필요한 네 가지 핵심 요소  \n\n| 요소 | 설명 | 비유 |\n|------|------|------|\n| **데이터** | 모델이 보는 예시들 (입력 + 정답 라벨) | 교과서와 답안지 |\n| **모델 구조** | 입력을 받아서 계산하고 결과를 내는 ‘뇌’(신경망) | 퍼즐 조각을 연결하는 틀 |\n| **손실 함수(Loss)** | 모델이 내놓은 답과 정답 사이의 차이를 숫자로 나타냄 | 시험 점수(오답이면 점수가 낮음) |\n| **최적화 알고리즘** | 손실을 줄이기 위해 모델 내부 가중치를 조금씩 바꾸는 방법 | 틀을 맞추기 위해 퍼즐 조각을 조금씩 움직임 |\n\n---\n\n## 3. 학습 과정 (한 번을 “에포크”라고 부름)\n\n1. **데이터 준비**  \n   - 사진과 정답(고양이/고양이 아님) 쌍을 모아 **훈련 데이터**와 **검증 데이터**로 나눔.  \n   - 사진을 같은 크기로 맞추고, 색을 정규화하는 등 전처리를 함.\n\n2. **한 번에 몇 개씩(배치) 꺼내기**  \n   - 전체 데이터를 한 번에 다 쓰면 계산이 너무 무거우니, **배치**(예: 32장) 단위로 나눠서 사용.\n\n3. **예측 → 손실 계산**  \n   - 모델이 배치에 있는 사진을 보고 “고양이 확률”을 출력.  \n   - 출력과 정답을 비교해 **손실(Loss)**을 구함. (예: 교차 엔트로피 손실)\n\n4. **가중치 업데이트**  \n   - 손실이 큰 방향으로 *

In [6]:
print(result.content)

## 인공지능 모델(특히 딥러닝) 학습이 어떻게 이루어지는지 쉽게 풀어볼게요  

### 1. 목표는 “무엇을 알게 만들고 싶은가?”
예를 들어 **고양이 사진을 보면 “고양이”라고 말할 수 있게** 하고 싶다고 해봅시다.  
이때 모델에게 **‘입력(사진) → 출력(고양이/고양이 아님)’**이라는 관계를 배우게 해야 합니다.

---

## 2. 학습에 필요한 네 가지 핵심 요소  

| 요소 | 설명 | 비유 |
|------|------|------|
| **데이터** | 모델이 보는 예시들 (입력 + 정답 라벨) | 교과서와 답안지 |
| **모델 구조** | 입력을 받아서 계산하고 결과를 내는 ‘뇌’(신경망) | 퍼즐 조각을 연결하는 틀 |
| **손실 함수(Loss)** | 모델이 내놓은 답과 정답 사이의 차이를 숫자로 나타냄 | 시험 점수(오답이면 점수가 낮음) |
| **최적화 알고리즘** | 손실을 줄이기 위해 모델 내부 가중치를 조금씩 바꾸는 방법 | 틀을 맞추기 위해 퍼즐 조각을 조금씩 움직임 |

---

## 3. 학습 과정 (한 번을 “에포크”라고 부름)

1. **데이터 준비**  
   - 사진과 정답(고양이/고양이 아님) 쌍을 모아 **훈련 데이터**와 **검증 데이터**로 나눔.  
   - 사진을 같은 크기로 맞추고, 색을 정규화하는 등 전처리를 함.

2. **한 번에 몇 개씩(배치) 꺼내기**  
   - 전체 데이터를 한 번에 다 쓰면 계산이 너무 무거우니, **배치**(예: 32장) 단위로 나눠서 사용.

3. **예측 → 손실 계산**  
   - 모델이 배치에 있는 사진을 보고 “고양이 확률”을 출력.  
   - 출력과 정답을 비교해 **손실(Loss)**을 구함. (예: 교차 엔트로피 손실)

4. **가중치 업데이트**  
   - 손실이 큰 방향으로 **가중치**를 조금씩 바꿈.  
   - 보통 **경사 하강법(SGD, Adam 등)**이라는 알고리즘을 사용해 “가장 빠르게 손실을 줄일 수 있는 방향”을 찾

### 2) PromptTemplate + LLM

In [7]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("You are an expert in AI Expert. Answer the question.\
                                       <Question>: {input}에 대해 쉽게 설명해주세요.")

print(type(prompt))
print(prompt)

<class 'langchain_core.prompts.prompt.PromptTemplate'>
input_variables=['input'] input_types={} partial_variables={} template='You are an expert in AI Expert. Answer the question.                                       <Question>: {input}에 대해 쉽게 설명해주세요.'


In [8]:
# llm: 공통 설정 셀의 llm 사용
# chain 연결 (LCEL)
chain = prompt | llm
print(type(chain))

# chain 호출
result = chain.invoke({"input": "인공지능 모델의 학습 원리"})
print(type(result))
print(result)

<class 'langchain_core.runnables.base.RunnableSequence'>
<class 'langchain_core.messages.ai.AIMessage'>
content='### 인공지능 모델이 “학습”한다는 것은?\n\n쉽게 말하면 **컴퓨터에게 문제를 풀 방법을 스스로 찾게 하는 과정**입니다. 사람은 교과서와 선생님의 설명을 듣고 문제를 풀어보면서 “공부”하고, 틀리면 왜 틀렸는지 생각하면서 고쳐요.  \n인공지능도 비슷한 절차를 거칩니다. 아래에 그 흐름을 단계별로 풀어볼게요.\n\n---\n\n## 1️⃣ 데이터가 교재·연습문제\n\n- **입력(Feature)** : 모델이 보고 판단할 내용 (예: 사진, 텍스트, 센서값 등)  \n- **정답(Label)** : 우리가 모델에게 알려주고 싶은 정답 (예: 사진에 고양이/강아지, 문장의 감정, 주가 상승/하락)\n\n> **비유**: 교과서에 나오는 예제와 정답을 보는 것과 비슷합니다.  \n\n---\n\n## 2️⃣ 모델 = “학생”\n\n- **구조** : 뉴런(노드)와 연결(가중치)로 이루어진 **신경망**이 가장 흔하지만, 결정 트리, 서포트 벡터 머신 등도 “학생”이라고 생각하면 됩니다.  \n- **가중치** : 학생이 문제를 풀 때 쓰는 “공식”이나 “전략”에 해당합니다. 처음엔 무작위(전혀 모르는 상태)라서 답을 거의 못 맞춥니다.\n\n---\n\n## 3️⃣ 손실 함수(Loss) = “오답률”\n\n- 모델이 내놓은 답과 정답을 비교해서 **얼마나 틀렸는지**를 숫자로 나타낸 것이 손실 함수입니다.  \n- 예시)  \n  - **MSE(Mean Squared Error)** : 회귀(숫자 예측) 문제에서 “예측값과 실제값 차이의 제곱 평균”.  \n  - **Cross‑Entropy** : 분류(카테고리) 문제에서 “정답 확률이 얼마나 낮은가”를 측정.\n\n> **비유**: 시험을 보면 틀린 문제 수가 바로 “오답률”. 손실 함수는 그 오답률을 수

In [9]:
print(result.content)

### 인공지능 모델이 “학습”한다는 것은?

쉽게 말하면 **컴퓨터에게 문제를 풀 방법을 스스로 찾게 하는 과정**입니다. 사람은 교과서와 선생님의 설명을 듣고 문제를 풀어보면서 “공부”하고, 틀리면 왜 틀렸는지 생각하면서 고쳐요.  
인공지능도 비슷한 절차를 거칩니다. 아래에 그 흐름을 단계별로 풀어볼게요.

---

## 1️⃣ 데이터가 교재·연습문제

- **입력(Feature)** : 모델이 보고 판단할 내용 (예: 사진, 텍스트, 센서값 등)  
- **정답(Label)** : 우리가 모델에게 알려주고 싶은 정답 (예: 사진에 고양이/강아지, 문장의 감정, 주가 상승/하락)

> **비유**: 교과서에 나오는 예제와 정답을 보는 것과 비슷합니다.  

---

## 2️⃣ 모델 = “학생”

- **구조** : 뉴런(노드)와 연결(가중치)로 이루어진 **신경망**이 가장 흔하지만, 결정 트리, 서포트 벡터 머신 등도 “학생”이라고 생각하면 됩니다.  
- **가중치** : 학생이 문제를 풀 때 쓰는 “공식”이나 “전략”에 해당합니다. 처음엔 무작위(전혀 모르는 상태)라서 답을 거의 못 맞춥니다.

---

## 3️⃣ 손실 함수(Loss) = “오답률”

- 모델이 내놓은 답과 정답을 비교해서 **얼마나 틀렸는지**를 숫자로 나타낸 것이 손실 함수입니다.  
- 예시)  
  - **MSE(Mean Squared Error)** : 회귀(숫자 예측) 문제에서 “예측값과 실제값 차이의 제곱 평균”.  
  - **Cross‑Entropy** : 분류(카테고리) 문제에서 “정답 확률이 얼마나 낮은가”를 측정.

> **비유**: 시험을 보면 틀린 문제 수가 바로 “오답률”. 손실 함수는 그 오답률을 수학적으로 만든 점수표입니다.

---

## 4️⃣ 최적화(Optimization) = “공부 방법 찾기”

### 4‑1. **경사하강법(Gradient Descent)**
- 손실을 최소화하려면 가중치를 어떻게 바꿔야 할까? → 손실 함수의 **기울기(

### 3) PromptTemplate + LLM(invoke()) + StrOutputParser

In [10]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 컴포넌트 정의 (llm 은 공통 설정 셀의 llm 사용)
prompt = PromptTemplate.from_template("You are an expert in AI Expert. \
                                      Answer the question. <Question>: {input}에 대해 쉽게 설명해주세요.")

output_parser = StrOutputParser()

# 2. chain 생성 (LCEL)
chain = prompt | llm | output_parser
print(type(chain))

# 3. chain의 invoke 호출
result = chain.invoke({"input": "인공지능 모델의 학습 원리"})
print(type(result))
print(result)

<class 'langchain_core.runnables.base.RunnableSequence'>
<class 'langchain_core.messages.base.TextAccessor'>
### 인공지능 모델이 어떻게 “학습”하나요? – 쉬운 설명

#### 1. **학습은 ‘경험’으로부터 배우는 과정**
- **우리 사람도 경험을 통해 배우죠.** 예를 들어, 사과와 바나나를 여러 번 보고 “이건 사과, 저건 바나나”라고 구분하게 됩니다.
- 인공지능(AI) 모델도 **많은 데이터(경험)를 보여받으며** 스스로 규칙을 찾아냅니다.

#### 2. **핵심 요소 3가지**

| 요소 | 역할 | 쉬운 비유 |
|------|------|----------|
| **데이터** | 모델에게 보여줄 예시(입력)와 정답(라벨) | 선생님이 주는 문제와 정답지 |
| **모델(함수)** | 입력을 받아서 예측값을 내는 ‘수학 공식’ | 학생이 문제를 풀 때 쓰는 방법 |
| **손실(오차) 함수** | 예측값과 정답 사이의 차이를 숫자로 표현 | 시험 점수(오답이 많을수록 점수가 낮음) |

#### 3. **학습 흐름 (한 번의 ‘반복’ = 에포크)**

1. **입력 → 예측**  
   - 모델에 데이터를 넣으면, 현재 가지고 있는 ‘규칙’(가중치)으로 예측값을 만든다.  
   - 예) 사진을 넣으면 “고양이” 혹은 “강아지”라고 말한다.

2. **오차 계산**  
   - 예측값과 정답을 비교해 손실 함수를 사용해 **얼마나 틀렸는지** 숫자로 구한다.  
   - 오차가 클수록 “아직 배운 게 부족해!” 라는 신호.

3. **가중치 조정 (최적화)**  
   - **‘경사 하강법(Gradient Descent)’**이라는 방법으로 오차를 줄이도록 가중치를 살짝씩 바꾼다.  
   - 비유: 산을 내려가듯이 “오차가 가장 크게 줄어드는 방향”으로 조금씩 움직인다.

4. **반복**  
   - 위 과정을 수천·수만 번 반복하면서 가중치가 점점 더

### 4) PromptTemplate + LLM(stream()) + StrOutputParser

In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 컴포넌트 정의 (llm 은 공통 설정 셀의 llm 사용)
prompt = PromptTemplate.from_template("You are an expert in AI Expert. \
                                      Answer the question. <Question>: {input}에 대해 쉽게 설명해주세요.")

# chain 연결 (LCEL)
chain = prompt | llm | StrOutputParser()

# 스트리밍 출력을 위한 요청
answer = chain.stream({"input": "인공지능 모델의 학습 원리"})
# 스트리밍 출력
#print(answer)

for token in answer:
    # 스트림에서 받은 데이터의 내용을 출력합니다. 줄바꿈 없이 이어서 출력하고, 버퍼를 즉시 비웁니다.
    print(token, end="", flush=True)

### 인공지능 모델이 어떻게 “배우는지” 쉽게 풀어 설명하면

1️⃣ **목표가 뭔가요?**  
- 모델은 **입력(예: 사진, 문장, 숫자)**을 받아서 **출력(예: 고양이/개 구분, 다음 단어 예측, 가격 예측)**을 만들어요.  
- 우리가 원하는 출력과 실제 정답(라벨) 사이의 차이를 **오차(손실, loss)** 라고 부릅니다.  

2️⃣ **학습은 “오차를 줄이는 연습”**  
- 마치 시험을 보면서 틀린 문제를 고쳐가며 실력을 올리는 과정과 비슷해요.  
- 모델은 처음엔 무작위로 **가중치(weight)** 라는 숫자들을 가지고 시작합니다. 이 가중치가 바로 “뇌의 시냅스” 역할을 해요.

3️⃣ **오차를 측정한다 → 손실 함수**  
- 모델이 만든 출력과 정답을 비교해서 **손실 함수**(예: 평균 제곱 오차, 교차 엔트로피)로 숫자 하나로 요약합니다.  
- 손실이 클수록 “아직 많이 틀렸어!” 라는 신호가 됩니다.

4️⃣ **오차를 줄이는 방향을 찾는다 → 미분과 기울기(Gradient)**  
- 수학에서 **미분**은 “어디를 조금 바꾸면 손실이 가장 많이 줄어들까?”를 알려주는 도구예요.  
- 손실 함수를 가중치 각각에 대해 미분하면 **기울기(gradient)** 가 나오고, 이 기울기가 “내가 지금 얼마나 잘못됐는지”를 알려줍니다.

5️⃣ **가중치를 조금씩 바꾼다 → 경사하강법(Gradient Descent)**  
- 기울기가 가리키는 방향과 반대로 가중치를 **조금씩(learning rate 라는 작은 step)** 이동시킵니다.  
- 이렇게 하면 손실이 조금씩 줄어들고, 모델이 점점 더 정확해져요.  
- 이 과정을 **에폭(epoch)** 라는 단위로 여러 번 반복합니다. (전체 데이터셋을 한 번 다 사용하면 한 에폭)

6️⃣ **반복 → 학습 완료**  
- 손실이 충분히 작아지거나, 검증 데이터에서 더 이상 성능이 향상되지 않을 때 학습을 멈춥니다.  
- 최종적으로 얻은 가중치가 바로 “배운 지식”이

##### 2) Multiple Chains
* Multi Chain을 활용한 영화 추천 및 줄거리 요약 

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
prompt1 = ChatPromptTemplate.from_template("{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.")

# Step 2: 추천된 영화의 줄거리를 요약
prompt2 = ChatPromptTemplate.from_template("{movie} 추천한 영화의 제목을 먼저 알려주시고,\
                                            줄을 바꾸어서 영화의 정보(제목,감독,캐스팅,줄거리)를 알려 주세요")

# 영화 예제용 모델 (공통 설정 셀의 get_llm 사용)
llm = get_llm(GROQ_MODEL)

# 체인 1: 영화 추천 (입력: 장르 → 출력: 영화 제목)
chain1 = prompt1 | llm | StrOutputParser()

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
movie = chain1.invoke({"genre": "Drama"})  # 영화 제목 얻기

print(type(movie))
print(" ===> 추천된 영화:")  # movie 값 출력
print(movie)


In [ ]:
# 체인 2: 줄거리 요약 (입력: 영화 제목 → 출력: 줄거리)
chain2 = (
    {"movie": chain1}  # chain1의 출력을 movie 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)
print(chain2)

# 실행: "SF" 장르의 영화 추천 및 줄거리 요약
response = chain2.invoke({"genre": "Drama"})
print("\n🔹 영화 줄거리 요약:\n", response) 

##### chain1과 chain2에서 영화 제목이 일관되게 전달 되도록 변경 

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from pprint import pprint

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
prompt1 = ChatPromptTemplate.from_template("{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.")

# Step 2: 추천된 영화의 줄거리를 요약
prompt2 = ChatPromptTemplate.from_template("{movie} 추천한 영화의 제목을 먼저 알려주시고, 줄을 바꾸어서 영화의 정보(제목,감독,캐스팅,줄거리)를 알려 주세요.")

llm = get_llm(GROQ_MODEL)

# 체인 1: 영화 추천 (입력: 장르 → 출력: 영화 제목)
chain1 = prompt1 | llm | StrOutputParser()

# 체인 2: 줄거리 요약 (입력: 영화 제목 → 출력: 줄거리)
chain2 = (
    {"movie": chain1}  # chain1의 출력을 movie 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)

# 실행: "Drama" 장르의 영화 추천 및 줄거리 요약
response = chain2.invoke({"genre": "Drama"})
print("\n🔹 영화 줄거리 요약:")
pprint(response)

In [ ]:
# 모델 설정 (공통 설정 셀의 get_llm 사용)
model = get_llm(GROQ_MODEL)

# 멀티 체인 실행 함수
def movie_chain(genre):
    print(f"\n🔹 1단계: '{genre}' 장르 영화 추천")
    
    # 체인 1: 영화 추천
    response1 = model.invoke(f"{genre} 장르에서 유명한 영화 제목 하나만 알려줘")
    movie = response1.content.strip()
    print(f"   → 추천 영화: {movie}")
    
    print(f"\n🔹 2단계: '{movie}' 영화 정보 조회")
    
    # 체인 2: 영화 정보
    response2 = model.invoke(f"{movie} 영화에 대해 간단히 알려줘. 줄거리, 감독, 출연진 포함해서")
    info = response2.content
    print(f"   → 영화 정보:\n{info}")
    
    return movie, info

# 실행
genre = input("원하는 장르를 입력하세요: ")
movie, info = movie_chain(genre)
print(f"\n✅ 최종 결과: '{movie}' 영화가 추천되었습니다!")